# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [ ]:
NAMES = "Audrey, Claire, Jonathan"

In [ ]:
!pip install torch scikit-learn pandas matplotlib

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

For binary classification, we would use binary cross-entropy as the cost function, while multi-class classification would use categorical cross-entropy. For regression, we would use mean squared error (MSE) to measure the difference between predicted and actual values.

The output layer would be different for each task. Binary classification would have one output unit with a sigmoid activation function to predict a probability between 0 and 1. Multi-class classification would have one output unit per class with a softmax activation function to produce class probabilities. Regression would have one output unit with a linear activation function to predict a continuous numerical value.

Standardizing the data is important because it puts the features on a similar scale, helping the model train more efficiently and preventing features with larger values from dominating the learning process. We should fit the scaler only on the training data to avoid data leakage.

We could detect overfitting by comparing training and holdout performance over multiple epochs. If training loss continues to decrease while holdout loss begins to increase, the model may be overfitting. Similarly, if training accuracy is high but holdout accuracy is much lower, the model may not generalize well to unseen data.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class BinaryMLP:
    def __init__(self, n_inputs, n_hidden, n_layers=1, activation="relu"):
        self.n_layers = n_layers
        self.activation = activation

        self.W1 = (torch.randn(n_inputs, n_hidden) / n_inputs ** 0.5).requires_grad_()
        self.b1 = torch.zeros(n_hidden, requires_grad=True)
        self.params = [self.W1, self.b1]

        if n_layers == 2:
            self.W2 = (torch.randn(n_hidden, n_hidden) / n_hidden ** 0.5).requires_grad_()
            self.b2 = torch.zeros(n_hidden, requires_grad=True)
            self.params += [self.W2, self.b2]

        self.W_out = (torch.randn(n_hidden, 1) / n_hidden ** 0.5).requires_grad_()
        self.b_out = torch.zeros(1, requires_grad=True)
        self.params += [self.W_out, self.b_out]

    def act(self, z):
        if self.activation == "relu":
            return torch.relu(z)
        return torch.tanh(z)

    def forward(self, X):
        h = self.act(X @ self.W1 + self.b1)
        if self.n_layers == 2:
            h = self.act(h @ self.W2 + self.b2)
        return torch.sigmoid(h @ self.W_out + self.b_out)

    def cost(self, y_pred, y):
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        return -torch.mean(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))

    def fit(self, X, y, X_hold, y_hold, epochs=100, lr=0.01):
        self.train_costs = []
        self.hold_costs = []
        for epoch in range(epochs):
            for i in torch.randperm(len(X)).tolist():
                loss = self.cost(self.forward(X[i:i + 1]), y[i:i + 1])
                loss.backward()
                with torch.no_grad():
                    for p in self.params:
                        p -= lr * p.grad
                        p.grad.zero_()
            with torch.no_grad():
                self.train_costs.append(self.cost(self.forward(X), y).item())
                self.hold_costs.append(self.cost(self.forward(X_hold), y_hold).item())

    def predict(self, X):
        with torch.no_grad():
            return (self.forward(X) >= 0.5).float()

In [ ]:
# 80/20 split
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_hold = scaler.transform(X_hold)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_hold = torch.tensor(X_hold, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_hold = torch.tensor(y_hold, dtype=torch.float32).reshape(-1, 1)

print(X_train.shape, X_hold.shape)

### Training and evaluation

We used a learning rate of 0.01 and 100 epochs for every model. Each update uses one random training point (stochastic gradient descent), so 100 epochs is 80,000 updates per model, which takes several minutes for all 16 models. We picked 100 epochs because the learning curves below show that the cost has pretty much flattened out by around epoch 40. We tried:

- One hidden layer with 2, 4, 8, 16 and 32 units
- Two hidden layers with 2, 8 and 32 units in each layer, to compare with one layer
- All of the above with ReLU and then with Tanh

In [ ]:
configs = [(1, 2), (1, 4), (1, 8), (1, 16), (1, 32), (2, 2), (2, 8), (2, 32)]  # (hidden layers, units per layer)

models = {}
results = []
for activation in ["relu", "tanh"]:
    for n_layers, n_hidden in configs:
        torch.manual_seed(42)  # same random start for every model
        model = BinaryMLP(n_inputs=2, n_hidden=n_hidden, n_layers=n_layers, activation=activation)
        model.fit(X_train, y_train, X_hold, y_hold, epochs=100, lr=0.01)
        models[(activation, n_layers, n_hidden)] = model
        results.append({
            "activation": activation,
            "hidden layers": n_layers,
            "units per layer": n_hidden,
            "train accuracy": (model.predict(X_train) == y_train).float().mean().item(),
            "holdout accuracy": (model.predict(X_hold) == y_hold).float().mean().item(),
            "train cost": model.train_costs[-1],
            "holdout cost": model.hold_costs[-1],
        })

results = pd.DataFrame(results)
results.round(3)

In [ ]:
# why did ReLU with two layers of 2 units do so badly? check how often each unit in layer 2 is not 0
m = models[("relu", 2, 2)]
with torch.no_grad():
    h1 = torch.relu(X_train @ m.W1 + m.b1)
    h2 = torch.relu(h1 @ m.W2 + m.b2)
print("share of training points where each layer 2 unit is not 0:", (h2 > 0).float().mean(dim=0).numpy())

In [ ]:
# training and holdout cost after each epoch
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, key in zip(axes.ravel(), [("relu", 1, 2), ("relu", 1, 32), ("relu", 2, 32),
                                  ("tanh", 1, 2), ("tanh", 1, 32), ("tanh", 2, 32)]):
    ax.plot(models[key].train_costs, label="train")
    ax.plot(models[key].hold_costs, label="holdout")
    ax.set_title(f"{key[0]}, {key[1]} layer(s), {key[2]} units")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cost")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# decision boundaries on the holdout set
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, key in zip(axes.ravel(), [("relu", 1, 2), ("relu", 1, 8), ("relu", 1, 32), ("relu", 2, 32),
                                  ("tanh", 1, 2), ("tanh", 1, 8), ("tanh", 1, 32), ("tanh", 2, 32)]):
    zz = models[key].predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_hold[:, 0], X_hold[:, 1], c=y_hold[:, 0], cmap="coolwarm", s=10)
    ax.set_title(f"{key[0]}, {key[1]} layer(s), {key[2]} units")
plt.tight_layout()
plt.show()

**How does the number of hidden units affect performance?**

The number of hidden units made a big difference at first and then stopped mattering. With only 2 units, both ReLU and Tanh got around 86% to 87% on the training set and about 85% on the holdout set. You can see why in the decision boundary plots, since with 2 units the boundary is basically a straight line, so it cuts right through both moons. With 4 units Tanh already jumped to about 97% training and 98% holdout accuracy, but ReLU was still at 88.9% and 86.5%. Once we got to 8 units, both activations were around 97% on the training set and 98.5% to 99% on the holdout set, and the boundary curves around the two moons. Going from 8 to 16 to 32 units didn't really change anything, because 8 units are already enough for this shape and the points that are still wrong are noisy points sitting inside the other moon.

We also didn't see any overfitting, even with 32 units. In the learning curves the holdout cost never starts going back up, and for the models with 8 or more units the holdout accuracy is actually a little higher than the training accuracy. That is just because the holdout set is small (200 points) and happened to get fewer of those noisy points.

**What changes when using two layers instead of one?**

When each layer already had enough units (8 or 32), adding a second layer gave pretty much the same results. For example, with 32 units the holdout accuracy was 98.0% with two layers vs 98.5% with one layer for ReLU, and 99.0% for both with Tanh. The boundaries with two layers also look almost the same as with one layer. The main difference we saw was in the learning curves. The two-layer models drop much faster at the start (they get down to a cost of about 0.1 in around 10 epochs instead of 20 or more), but they also jump around a lot more from epoch to epoch, since there are a lot more weights getting updated by one noisy example at a time. So for a simple 2D problem like this, a second layer doesn't really help, because one layer can already make the shape we need.

With only 2 units per layer, the second layer actually made things worse for ReLU. The ReLU model with two layers of 2 units got stuck at 50% accuracy, which is the same as guessing, and its cost was 0.693 (log 2), which means it predicts about 0.5 for every point. The check we ran shows that both units in its second layer output 0 for every training point. When a ReLU unit outputs 0 for everything, its gradient is also 0, so SGD can never update it again and it stays "dead". The Tanh model with two layers of 2 units didn't have this problem and got 83.5% on the holdout set, which is about the same as one layer of 2 units.

**How does the activation function (ReLU vs. Tanh) influence results?**

When there were enough units (8 or more), ReLU and Tanh ended up with almost the same accuracy (about 97% training and 98% to 99% holdout), so the activation function mostly changed the shape of the boundary and how reliable the training was. ReLU is a straight line on each side of 0, so the ReLU boundaries are made of straight pieces with sharp corners, which you can see in the plots. On the other hand, Tanh is smooth, so the Tanh boundaries are smooth curves that follow the moons more naturally.

The bigger difference was with the small networks. Tanh did a lot better with 4 units (98.0% vs 86.5% holdout) and with two layers of 2 units (83.5% vs 50%). With 4 ReLU units there are only a few straight pieces to build the boundary from, and in the two-layer case the ReLU units died, while a Tanh unit always passes some gradient back so it keeps learning. Overall, ReLU was fine when the network had plenty of units, but Tanh was more reliable when the network was really small. Since we only trained each model once with one random seed, the results for the really small networks could change with a different starting point, but the results with 8 or more units were about the same for every setup we tried.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [ ]:
%pip install torch scikit-learn numpy pandas matplotlib

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder


class MultiClassMLP:
    def __init__(self, n_inputs, n_hidden, n_classes, n_layers=1, activation="relu"):
        self.n_layers = n_layers
        self.activation = activation

        self.W1 = (torch.randn(n_inputs, n_hidden) / n_inputs ** 0.5).requires_grad_()
        self.b1 = torch.zeros(n_hidden, requires_grad=True)
        self.params = [self.W1, self.b1]

        if n_layers == 2:
            self.W2 = (torch.randn(n_hidden, n_hidden) / n_hidden ** 0.5).requires_grad_()
            self.b2 = torch.zeros(n_hidden, requires_grad=True)
            self.params += [self.W2, self.b2]

        # one output unit per class
        self.W_out = (torch.randn(n_hidden, n_classes) / n_hidden ** 0.5).requires_grad_()
        self.b_out = torch.zeros(n_classes, requires_grad=True)
        self.params += [self.W_out, self.b_out]

    def act(self, z):
        if self.activation == "relu":
            return torch.relu(z)
        return torch.tanh(z)

    def forward(self, X):
        h = self.act(X @ self.W1 + self.b1)
        if self.n_layers == 2:
            h = self.act(h @ self.W2 + self.b2)
        # softmax turns the scores into probabilities that sum to 1 across the classes
        return F.softmax(h @ self.W_out + self.b_out, dim=1)

    def cost(self, y_pred, y):
        # categorical cross-entropy: y is one-hot, so only the true class's log probability counts
        y_pred = torch.clamp(y_pred, 1e-7, 1)
        return -torch.mean(torch.sum(y * torch.log(y_pred), dim=1))

    def fit(self, X, y, X_hold, y_hold, epochs=100, lr=0.01):
        self.train_costs = []
        self.hold_costs = []
        for epoch in range(epochs):
            for i in torch.randperm(len(X)).tolist():
                loss = self.cost(self.forward(X[i:i + 1]), y[i:i + 1])
                loss.backward()
                with torch.no_grad():
                    for p in self.params:
                        p -= lr * p.grad
                        p.grad.zero_()
            with torch.no_grad():
                self.train_costs.append(self.cost(self.forward(X), y).item())
                self.hold_costs.append(self.cost(self.forward(X_hold), y_hold).item())

    def predict(self, X):
        # predicted class = the output unit with the highest probability
        with torch.no_grad():
            return torch.argmax(self.forward(X), dim=1)

In [ ]:
# 80/20 split (stratified so each class has the same share in both sets)
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_hold = scaler.transform(X_hold)

# one-hot encode the labels for the cost, e.g. 2 -> [0, 0, 1]
encoder = OneHotEncoder(sparse_output=False)
Y_train = encoder.fit_transform(y_train.reshape(-1, 1))
Y_hold = encoder.transform(y_hold.reshape(-1, 1))

X_train = torch.tensor(X_train, dtype=torch.float32)
X_hold = torch.tensor(X_hold, dtype=torch.float32)
Y_train = torch.tensor(Y_train, dtype=torch.float32)
Y_hold = torch.tensor(Y_hold, dtype=torch.float32)
# keep the plain class labels too, to compute accuracy
y_train = torch.tensor(y_train)
y_hold = torch.tensor(y_hold)

print(X_train.shape, Y_train.shape, X_hold.shape, Y_hold.shape)

In [ ]:
configs = [(1, 2), (1, 4), (1, 8), (1, 16), (1, 32), (2, 2), (2, 8), (2, 32)]  # (hidden layers, units per layer)

models = {}
results = []
for activation in ["relu", "tanh"]:
    for n_layers, n_hidden in configs:
        torch.manual_seed(42)  # same random start for every model
        model = MultiClassMLP(n_inputs=2, n_hidden=n_hidden, n_classes=3,
                              n_layers=n_layers, activation=activation)
        model.fit(X_train, Y_train, X_hold, Y_hold, epochs=100, lr=0.01)
        models[(activation, n_layers, n_hidden)] = model
        results.append({
            "activation": activation,
            "hidden layers": n_layers,
            "units per layer": n_hidden,
            "train accuracy": (model.predict(X_train) == y_train).float().mean().item(),
            "holdout accuracy": (model.predict(X_hold) == y_hold).float().mean().item(),
            "train cost": model.train_costs[-1],
            "holdout cost": model.hold_costs[-1],
        })

results = pd.DataFrame(results)
results.round(3)

In [ ]:
# check the ReLU model with two layers of 2 units for "dead" units, like in the binary task
m = models[("relu", 2, 2)]
with torch.no_grad():
    h1 = torch.relu(X_train @ m.W1 + m.b1)
    h2 = torch.relu(h1 @ m.W2 + m.b2)
print("share of training points where each layer 2 unit is not 0:", (h2 > 0).float().mean(dim=0).numpy())

In [ ]:
# training and holdout cost after each epoch
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, key in zip(axes.ravel(), [("relu", 1, 2), ("relu", 1, 32), ("relu", 2, 32),
                                  ("tanh", 1, 2), ("tanh", 1, 32), ("tanh", 2, 32)]):
    ax.plot(models[key].train_costs, label="train")
    ax.plot(models[key].hold_costs, label="holdout")
    ax.set_title(f"{key[0]}, {key[1]} layer(s), {key[2]} units")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cost")
    ax.legend()
plt.tight_layout()
plt.show()

# decision boundaries on the holdout set
xx, yy = np.meshgrid(np.linspace(-3.75, 3.75, 200), np.linspace(-3.75, 3.75, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, key in zip(axes.ravel(), [("relu", 1, 2), ("relu", 1, 8), ("relu", 1, 32), ("relu", 2, 32),
                                  ("tanh", 1, 2), ("tanh", 1, 8), ("tanh", 1, 32), ("tanh", 2, 32)]):
    zz = models[key].predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, cmap="brg")
    ax.scatter(X_hold[:, 0], X_hold[:, 1], c=y_hold, cmap="brg", s=10)
    ax.set_title(f"{key[0]}, {key[1]} layer(s), {key[2]} units")
plt.tight_layout()
plt.show()

### Reflection and discussion
#### How does the number of hidden units affect performance?

The number of hidden units did not matter much here. With just 2 units, the models already got about 78% right on the training data and 82.5% on the holdout data. The best model (Tanh, one layer, 16 units) only got 85.5% on the holdout data. This is very different from the binary task, where more units helped a lot (from about 85% to 98%).

The reason is the data. The dataset has 15% of its labels flipped at random (flip_y=0.15), and two of the classes overlap. No model can get those points right, so even a tiny model is close to the best we can do.

More units mostly made the boundaries messier, not better. With 2 units, the boundaries are almost straight lines. With 32 units, odd extra regions show up where there is no data, like the small island in the top-left corner for ReLU with 32 units. That model is trying to fit the noisy points, and it actually did worse on the holdout data (81.5%).

The holdout accuracy was a little higher than the training accuracy for every model. This is probably just luck: the holdout set is small (200 points) and likely got fewer of the flipped labels.

### What changes when using two layers instead of one?

Two layers fit the training data a bit better but did not do better on new data. With 32 units, the two-layer ReLU model had the lowest training cost (0.518 vs 0.584 for one layer), but its holdout cost was higher (0.579 vs 0.574). In its learning curve, the training cost keeps going down, but the holdout cost is lowest around epoch 27 and then goes back up. This is a sign of overfitting: the extra layer starts learning the noise. The two-layer Tanh models show the same thing. The accuracy numbers don't show this as clearly as the cost does.

With 2 units per layer, ReLU had a "dead" unit again. One of the two units in the second layer outputs 0 for every point, so it stops learning. Unlike the binary task, the other unit still worked, so the model still got 82.5% on the holdout data. But it had the worst cost of all the models (0.715 training, 0.692 holdout), because it was much less confident in its answers.

#### How does the activation function (ReLU vs. Tanh) influence results?

ReLU and Tanh gave almost the same accuracy. The differences were only 1–2%, which is just 2–4 points out of 200. Tanh did a little better in most cases (for example, 85.5% vs 84.0% with 16 units), and it never had dead units. The main difference was the shape of the boundaries. ReLU makes boundaries out of straight lines with sharp corners, while Tanh makes smooth curves.

Overall, for noisy data like this, a small model with one layer of about 8–16 units works as well as anything bigger. We only trained each model once, so differences of 1–2% could change with a different random start.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
#Implementation
class RegressionMLP:

    def __init__(self, n_hidden, n_hidden_2=None, learning_rate=0.01, n_features=2, activation="relu", random_state=42):
        assert activation in ("relu", "tanh")

        self.n_hidden = n_hidden
        self.n_hidden_2 = n_hidden_2
        self.learning_rate = learning_rate
        self.n_features = n_features
        self.activation = activation
        self.random_state = random_state

        # Reproducible initialization
        generator = torch.Generator()
        generator.manual_seed(random_state)

        # Initialize first hidden layer
        gain = 2.0 if activation == "relu" else 1.0

        self.W1 = torch.randn(n_features, n_hidden, generator=generator) * np.sqrt(gain / n_features)
        self.b1 = torch.zeros(1, n_hidden)

        if n_hidden_2 is None:
            # One hidden layer -> output layer
            self.W2 = torch.randn(n_hidden, 1, generator=generator) / np.sqrt(n_hidden)
            self.b2 = torch.zeros(1, 1)

        else:
            # Two hidden layers -> output layer
            self.W2 = torch.randn(n_hidden, n_hidden_2, generator=generator) * np.sqrt(gain / n_hidden)
            self.b2 = torch.zeros(1, n_hidden_2)
            self.W3 = torch.randn(n_hidden_2, 1, generator=generator) / np.sqrt(n_hidden_2)
            self.b3 = torch.zeros(1, 1)

    # Hidden-layer activation
    def _act(self, Z):
        if self.activation == "relu":
            return torch.clamp(Z, min=0)
        return torch.tanh(Z)

    # Derivative of the hidden-layer activation
    def _act_grad(self, Z, A):
        if self.activation == "relu":
            return (Z > 0).float()
        return 1.0 - A**2

    #forward propagation
    def forward(self, X):
        Z1 = X @ self.W1 + self.b1
        A1 = self._act(Z1)

        if self.n_hidden_2 is None:
            # One hidden layer
            y_pred = A1 @ self.W2 + self.b2
            self.cache = {"X": X, "Z1": Z1, "A1": A1, "y_pred": y_pred}

        else:
            # Two hidden layers
            Z2 = A1 @ self.W2 + self.b2
            A2 = self._act(Z2)
            y_pred = A2 @ self.W3 + self.b3

            self.cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "y_pred": y_pred}
        return y_pred

    # Cost: half mean squared error
    def cost(self, y, y_pred):
        return torch.mean((y.reshape(-1, 1) - y_pred)**2) / 2

    # Manual backpropagation
    def _backward(self, y):
        c = self.cache

        X = c["X"]
        Z1, A1 = c["Z1"], c["A1"]
        y_pred = c["y_pred"]

        m = X.shape[0]
        y = y.reshape(-1, 1)

        # Gradient at output
        dZ_out = (y_pred - y) / m

        if self.n_hidden_2 is None:
            # Output layer gradients
            dW2 = A1.T @ dZ_out
            db2 = dZ_out.sum(dim=0, keepdim=True)

            # First hidden layer gradients
            dZ1 = (dZ_out @ self.W2.T) * self._act_grad(Z1, A1)

            dW1 = X.T @ dZ1
            db1 = dZ1.sum(dim=0, keepdim=True)

            return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

        # Second hidden layer and output layer gradients
        Z2, A2 = c["Z2"], c["A2"]

        dW3 = A2.T @ dZ_out
        db3 = dZ_out.sum(dim=0, keepdim=True)

        dZ2 = (dZ_out @ self.W3.T) * self._act_grad(Z2, A2)

        dW2 = A1.T @ dZ2
        db2 = dZ2.sum(dim=0, keepdim=True)

        # First hidden layer gradients
        dZ1 = (dZ2 @ self.W2.T) * self._act_grad(Z1, A1)

        dW1 = X.T @ dZ1
        db1 = dZ1.sum(dim=0, keepdim=True)

        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}

    # Training with stochastic gradient descent 
    def fit(self, X, y, epochs=150, X_val=None, y_val=None):
        y = y.reshape(-1, 1)
        m = X.shape[0]

        rng = np.random.default_rng(self.random_state)

        self.cost_history = []
        self.val_cost_history = []

        for epoch in range(epochs):

            # Shuffle observations each epoch
            indices = rng.permutation(m)

            #update weights after each observation
            for i in indices:
                xi = X[i:i+1]
                yi = y[i:i+1]

                self.forward(xi)
                grads = self._backward(yi)

                # Manual parameter updates
                with torch.no_grad():
                    for name, grad in grads.items():
                        parameter = getattr(self, name)
                        parameter -= self.learning_rate * grad

            # Record training cost after each epoch
            with torch.no_grad():
                train_pred = self.forward(X)
                self.cost_history.append(self.cost(y, train_pred).item())

                # Holdout cost is for evaluation only
                if X_val is not None and y_val is not None:
                    val_pred = self.forward(X_val)
                    self.val_cost_history.append(self.cost(y_val, val_pred).item())

        return self

    # Prediction
    def predict(self, X):
        with torch.no_grad():
            return self.forward(X).reshape(-1)

In [ ]:
#Data Preparation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32)

In [ ]:
#Model Traning and Evaluation
from sklearn.metrics import mean_squared_error
configs = [(1, 2), (1, 4), (1, 8), (1, 16), (1, 32), (2, 2), (2, 4), (2, 8), (2, 16), (2, 32)]  # (hidden layers, units per layer)

models = {}
results = []

for activation in ["relu", "tanh"]:
    for n_layers, n_hidden in configs:
            model = RegressionMLP(n_hidden=n_hidden, 
                                  n_hidden_2=n_hidden if n_layers==2 else None, 
                                  learning_rate=0.01, 
                                  n_features=X_train_t.shape[1], 
                                  activation=activation, 
                                  random_state=42)
            model.fit(X_train_t, y_train_t, epochs=150, X_val=X_test_t, y_val=y_test_t)
            models[(activation, n_layers, n_hidden)] = model

            train_pred_scaled = model.predict(X_train_t).numpy()
            hold_pred_scaled = model.predict(X_test_t).numpy()

            train_pred = scaler_y.inverse_transform(train_pred_scaled.reshape(-1,1)).ravel()
            hold_pred = scaler_y.inverse_transform(hold_pred_scaled.reshape(-1,1)).ravel()

            results.append({
                  "activation": activation,
                  "hidden layers": n_layers,
                  "units per layers": n_hidden,
                  "train MSE": mean_squared_error(y_train, train_pred),
                  "holdout MSE": mean_squared_error(y_test, hold_pred),
                  "train cost": model.cost_history[-1],
                  "holdout cost": model.val_cost_history[-1]
            })

results = pd.DataFrame(results)
results.round(3)

In [ ]:
# training and holdout cost after each epoch
fig, axes = plt.subplots(5, 4, figsize=(18, 20), sharey=True)

keys = [
    (activation, n_layers, n_hidden)
    for activation in ["relu", "tanh"]
    for n_layers, n_hidden in configs
]

for ax, key in zip(axes.ravel(), keys):
    ax.plot(models[key].train_costs, label="train")
    ax.plot(models[key].hold_costs, label="holdout")
    ax.set_title(f"{key[0]}, {key[1]} layer(s), {key[2]} units")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cost")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
summary = (results.groupby(["hidden layers", "units per layers", "activation"])["holdout MSE"].mean().reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, layers in zip(axes, [1, 2]):
    subset = summary[summary["hidden layers"] == layers]

    for activation in ["relu", "tanh"]:
        data = subset[subset["activation"] == activation].sort_values("units per layers")
        ax.plot(data["units per layers"], data["holdout MSE"],
                marker="o", label=activation)

    ax.set_title(f"{layers} Hidden Layer(s)")
    ax.set_xlabel("Units per Hidden Layer")
    ax.set_ylabel("Holdout MSE")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Effect of Hidden Units on Holdout MSE")
plt.tight_layout()
plt.show()

#### How does the number of hidden units affect performance?

The number of hidden units affected performance differently depending on the activation function and the number of hidden layers. For one hidden layer with ReLU, increasing the number of units from 2 to 8 reduced holdout MSE from approximately 5,770 to 5,505, making 8 units the best-performing width for this configuration. However, increasing the width to 16 or 32 units increased MSE again. With tanh and one hidden layer, holdout MSE generally increased as the number of units increased. For two hidden layers, increasing the number of units generally worsened performance, particularly with ReLU, whose MSE reached approximately 5,895 at 32 units. In contrast, tanh consistently achieved lower MSE than ReLU for the two-layer networks. Overall, adding more hidden units did not necessarily improve performance; the best number of units depended on the activation function and network depth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, activation in zip(axes, ["relu", "tanh"]):
    subset = summary[summary["activation"] == activation]

    for layers in [1, 2]:
        data = subset[subset["hidden layers"] == layers].sort_values("units per layers")
        ax.plot(data["units per layers"], data["holdout MSE"],
                marker="o", label=f"{layers} layer(s)")

    ax.set_title(f"{activation.capitalize()} Activation")
    ax.set_xlabel("Units per Hidden Layer")
    ax.set_ylabel("Holdout MSE")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Effect of Network Depth on Holdout MSE")
plt.tight_layout()
plt.show()

#### What changes when using two layers instead of one?

Using two hidden layers did not consistently improve performance. With ReLU, one hidden layer performed better at most tested widths, particularly at 8 and 32 units. With tanh, two layers performed better at 16 and 32 units, but slightly worse at 2 and 4 units. This suggests that adding depth can help in some configurations, but a deeper network is not automatically better. The effect depends on the activation function and the number of units.

In [ ]:
# Effect of activation function on holdout MSE
summary = (results.groupby(["hidden layers", "units per layers", "activation"])["holdout MSE"].mean().reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, layers in zip(axes, [1, 2]):
    subset = summary[summary["hidden layers"] == layers]

    for activation in ["relu", "tanh"]:
        data = subset[subset["activation"] == activation].sort_values("units per layers")
        ax.plot(data["units per layers"],data["holdout MSE"],marker="o", label=activation.capitalize())

    ax.set_title(f"{layers} Hidden Layer(s)")
    ax.set_xlabel("Units per Hidden Layer")
    ax.set_ylabel("Holdout MSE")
    ax.legend(title="Activation")
    ax.grid(True, alpha=0.3)

plt.suptitle("Effect of Activation Function on Holdout MSE")
plt.tight_layout()
plt.show()

#### How does the activation function (ReLU vs. Tanh) influence results?

The activation function influenced performance differently depending on the network architecture. With one hidden layer, ReLU generally performed better than tanh as the number of units increased.With two hidden layers, tanh generally performed better than ReLU, particularly at 32 units, where tanh had a holdout MSE of approximately 5,735 compared with roughly 5,895 for ReLU.

Overall, ReLU performed better for one-hidden-layer networks, while tanh performed better for two-hidden-layer networks in these experiments. This suggests that the best activation function depends on the network architecture rather than one function consistently outperforming the other.

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

Inheritance ould have helped us avoid repeating code across BinaryMLP, MultiClassMLP, and RegressionMLP by allowing the different classes to inherit common methods and attributes from a shared parent class, such as BaseMLP. Since all three models are multilayer perceptrons, they share much of the same structure, including weight and bias initialization, forward propagation through one or two hidden layers, and stochastic gradient descent updates during training.

For example, the parent class could implement __init__ to initialize the hidden layers and their weights and biases, as well as forward to pass the input through the network. It could also contain common training functionality in fit, including looping through individual training observations, calculating gradients, and updating parameters. The three subclasses could inherit these methods rather than repeating the same code.

Each subclass could inherit this shared code and define only what is different for its task. For example, BinaryMLP would use binary cross-entropy loss, MultiClassMLP would use multiclass cross-entropy loss, and RegressionMLP would use mean squared error.


This approach would reduce duplicated code, make the implementations easier to maintain, and allow improvements to the shared training algorithm to benefit all three models. It would also make the code easier to extend, since a new prediction task could be implemented by creating another subclass and defining its task-specific output and loss functions rather than building an entirely new MLP from scratch.

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

Our main challenge was that each group member mainly understood their own section of the neural network code rather than how all the components worked together. We divided up the implementation, and when we combined our work, it was sometimes difficult to understand how each section connected or troubleshoot errors outside our individual parts. For example, changes to one part of the model could affect another section, and we did not always know how to identify the source of the problem. For the next lab, we will take time to walk through the completed code together, have each person explain their section, and test the full implementation as a group so everyone understands how the pieces fit together.